# Module 4: Session Managers (10 min)

Add file-based persistence to the customer service agent. Stop the agent, restart it, and watch it remember the previous conversation.

**Prerequisites:** Modules 1-3 completed

In [ ]:
!pip install -q -r requirements.txt

---

## Part 1: Agent Without Persistence (The Problem)

By default, agents lose their memory when you recreate them.

In [2]:
from strands import Agent, AgentSkills
from customer_service_tools import lookup_customer, get_order_history, process_refund

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise.

If there are previous messages in the conversation history, use that context
to continue helping the customer without asking them to repeat information."""

# First interaction
agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)
agent("Hi, I'm customer C-1001. Can you look up my account?")

print(f"\n📝 Messages stored: {len(agent.messages)}")

Sure! Let me pull up your account right away.
Tool #1: lookup_customer
I found your account! Here are your details:

- **Name:** Sarah Johnson
- **Email:** sarah.johnson@email.com
- **Phone:** 555-0142
- **Account Status:** Active ✅

How can I help you today, Sarah?
📝 Messages stored: 4


In [3]:
# Simulate a "restart" — create a new agent instance
agent2 = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)

print(f"Messages after restart: {len(agent2.messages)}")
# The agent has no memory of the previous conversation!
agent2("What was my account status again?")

Messages after restart: 0
I don't have your customer information on file yet in our conversation. Could you please provide me with your **customer ID** so I can look up your account? It should be in the format **C-XXXX**.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I don't have your customer information on file yet in our conversation. Could you please provide me with your **customer ID** so I can look up your account? It should be in the format **C-XXXX**."}], 'metadata': {'usage': {'inputTokens': 809, 'outputTokens': 49, 'totalTokens': 858}, 'metrics': {'latencyMs': 1595, 'timeToFirstByteMs': 992}}, 'tracking_id': '98939006-54ce-4f82-bf78-2dda70d56b16'}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[1.777409315109253], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='5247f2ee-af5c-44b3-bae0-a7d651bdcfda', usage={'inputTokens': 809, 'outputTokens': 49, 'totalTokens': 858})], usage={'inputTokens': 809, 'outputTokens': 49, 'totalTokens': 858})], traces=[<strands.telemetry.metrics.Trace object at 0xffff4e0308d0>], accumulated_usage={'inputTokens': 809, 'outputTokens': 49, 'totalTokens': 858}, accumu

---

## Part 2: Add FileSessionManager

The `FileSessionManager` saves conversation history to disk. On restart, it reloads the messages.

The agent below uses **two** complementary pieces:
- **Session manager** (`FileSessionManager`) - persists the conversation *outside* the process so it survives restarts. This module's focus.
- **Conversation manager** (`SlidingWindowConversationManager`) - bounds what's kept *in context* on each call (here, the most recent 20 messages) so it doesn't grow without limit.

They solve different problems and work together: one stores history, the other trims what the model sees.

> **Tip:** The SDK also supports `context_manager="auto"`, which uses `SummarizingConversationManager` with proactive compression — a smarter alternative to sliding window. It's not yet in the official docs, but worth watching in the [Strands changelog](https://strandsagents.com/docs/changelog/).

In [4]:
from strands.session.file_session_manager import FileSessionManager
from strands.agent.conversation_manager import SlidingWindowConversationManager

# Clean up any previous session files
import shutil, os
if os.path.exists("./sessions"):
    shutil.rmtree("./sessions")

session_manager = FileSessionManager(
    session_id="customer-session-001",
    storage_dir="./sessions",
)

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[AgentSkills(skills=["./skills"])],
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=20),
    # Alternative: context_manager="auto" uses SummarizingConversationManager with
    # proactive compression — smarter than sliding window but not yet in the official docs.
    session_manager=session_manager,
)

# First interaction — this gets saved to disk
agent("Hi, I'm customer C-1001. Can you look up my account?")
print(f"\n📁 Session saved. Messages: {len(agent.messages)}")

Sure! Let me look up your account right away.
Tool #1: lookup_customer
I found your account! Here's a summary:

- **Name:** Sarah Johnson
- **Email:** sarah.johnson@email.com
- **Phone:** 555-0142
- **Account Status:** Active ✅

How can I help you today, Sarah?
📁 Session saved. Messages: 4


---

## Part 3: Restart and Remember

Create a brand new agent with the same session ID. It should remember everything.

In [5]:
# Simulate restart — new agent, same session_id
session_manager_2 = FileSessionManager(
    session_id="customer-session-001",
    storage_dir="./sessions",
)

agent_restarted = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[AgentSkills(skills=["./skills"])],
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=20),
    # Alternative: context_manager="auto" uses SummarizingConversationManager with
    # proactive compression — smarter than sliding window but not yet in the official docs.
    session_manager=session_manager_2,
)

print(f"🔄 Restored messages: {len(agent_restarted.messages)}")
print("The agent remembers the previous conversation!\n")

# Ask a follow-up — the agent should know we're C-1001
agent_restarted("What orders do I have? You should already know my customer ID.")

unable to find previously injected skills XML in system prompt, re-appending


🔄 Restored messages: 4
The agent remembers the previous conversation!

Of course! Let me pull up your order history right away.
Tool #1: get_order_history
Here are your orders, Sarah:

1. **ORD-5521** — Wireless Headphones — $79.99
   - Status: ✅ Delivered (April 28, 2025)
   - Tracking: TRK-998877

2. **ORD-5488** — USB-C Hub — $45.00
   - Status: 📦 Shipped (Est. Delivery: May 6, 2025)
   - Tracking: TRK-887766

Is there anything else I can help you with regarding these orders?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Here are your orders, Sarah:\n\n1. **ORD-5521** — Wireless Headphones — $79.99\n   - Status: ✅ Delivered (April 28, 2025)\n   - Tracking: TRK-998877\n\n2. **ORD-5488** — USB-C Hub — $45.00\n   - Status: 📦 Shipped (Est. Delivery: May 6, 2025)\n   - Tracking: TRK-887766\n\nIs there anything else I can help you with regarding these orders?'}], 'metadata': {'usage': {'inputTokens': 1511, 'outputTokens': 136, 'totalTokens': 1647}, 'metrics': {'latencyMs': 1720, 'timeToFirstByteMs': 866}}, 'tracking_id': 'f74c6369-93f3-4653-85ff-6763e7bac6ad'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'get_order_history': ToolMetrics(tool={'toolUseId': 'tooluse_CXEkXkkoqcEK7MPEZyHLUN', 'name': 'get_order_history', 'input': {'customer_id': 'C-1001'}}, call_count=1, success_count=1, error_count=0, total_time=0.0005738735198974609)}, cycle_durations=[1.5391290187835693, 1.7811899185180664], agent_invocations=[Ag

---

## 🎯 Try It Yourself

Check what's stored on disk:

In [6]:
import json

# FileSessionManager stores data in nested folders:
#   ./sessions/session_<id>/session.json
#   ./sessions/session_<id>/agents/agent_<id>/agent.json
#   ./sessions/session_<id>/agents/agent_<id>/messages/message_*.json
# So walk the tree recursively instead of listing only the top level.
session_dir = "./sessions"
for root, _dirs, files in os.walk(session_dir):
    for f in sorted(files):
        if f.endswith(".json"):
            filepath = os.path.join(root, f)
            rel = os.path.relpath(filepath, session_dir)
            print(f"📄 {rel} ({os.path.getsize(filepath)} bytes)")

# Count how many message files were persisted
message_files = [
    os.path.join(root, f)
    for root, _dirs, files in os.walk(session_dir)
    for f in files
    if f.startswith("message_") and f.endswith(".json")
]
print(f"\n💬 Messages persisted to disk: {len(message_files)}")

📄 session_customer-session-001/session.json (173 bytes)
📄 session_customer-session-001/agents/agent_default/agent.json (1322 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_0.json (360 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_1.json (793 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_2.json (595 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_3.json (735 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_4.json (370 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_5.json (806 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_6.json (735 bytes)
📄 session_customer-session-001/agents/agent_default/messages/message_7.json (888 bytes)

💬 Messages persisted to disk: 8


---

## 💬 Want a real multi-turn conversation?

In a notebook, each cell is a **single turn**. To chat back and forth with the **persistent** agent, run the companion script in a **terminal**. From the cloned repo:

```bash
cd samples/04-session-managers
pip install -r requirements.txt
python chat.py
```

Type your messages, and `quit` (or Ctrl+C) to exit. Because it uses `FileSessionManager`, this is also the persistence demo: quit and run `python chat.py` again — it restores the earlier conversation. Use `--session-id <name>` to keep separate sessions.

---

## What's Next

The agent is persistent and follows rules — it's complete enough to ship. In **Module 5: Deploy**, you'll package this same agent and deploy it to Amazon Bedrock AgentCore Runtime with a single CLI command.